#**Progetto Big Data - Analisi di Wikipedia**

Wikidata Insights, un'azienda leader nella gestione di contenuti digitali, è stata incaricata da Wikimedia per ottimizzare l'analisi e la categorizzazione dei contenuti di Wikipedia. Per supportare la loro continua espansione e migliorare l'organizzazione delle informazioni, Wikidata Insights ha deciso di condurre un progetto avanzato di data analysis e machine learning. L'obiettivo principale è comprendere meglio il vasto patrimonio di contenuti informativi offerti da Wikipedia e sviluppare un sistema di classificazione automatica che consenta di categorizzare efficacemente i nuovi articoli futuri.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim
spark = SparkSession.builder.appName("WikipediaAnalysis").getOrCreate()

# **1. CARICAMENTO DATI**

## **1.1 Caricamento Dati**

**Caricamento del Dataset da S3**:

---
In questa sezione carico il dataset wikipedia.csv direttamente dal link S3 indicato nell’enunciato. <br> Poiché Spark non può leggere un file remoto, lo scarico prima in locale tramite wget e poi lo importo come DataFrame Spark.

Il file contiene testi molto lunghi con virgolette e interruzioni di riga, quindi uso alcune opzioni aggiuntive per permettere a Spark di leggerlo correttamente.

In [2]:
url = "https://proai-datasets.s3.eu-west-3.amazonaws.com/wikipedia.csv"

# Spark può leggere direttamente da URL solo se il file è locale,
# quindi lo scarico prima con wget
!wget -O wikipedia.csv {url}

#Carico il file con Spark e creo un Dataframe con Spark
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("mode", "PERMISSIVE") \
    .csv("wikipedia.csv")

df.head(5)

--2026-02-19 21:13:10--  https://proai-datasets.s3.eu-west-3.amazonaws.com/wikipedia.csv
Resolving proai-datasets.s3.eu-west-3.amazonaws.com (proai-datasets.s3.eu-west-3.amazonaws.com)... 52.95.155.86, 3.5.205.57
Connecting to proai-datasets.s3.eu-west-3.amazonaws.com (proai-datasets.s3.eu-west-3.amazonaws.com)|52.95.155.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1003477941 (957M) [text/csv]
Saving to: ‘wikipedia.csv’

wikipedia.csv       100%[===================>] 956.99M  6.25MB/s    in 2m 6s   

2026-02-19 21:15:17 (7.57 MB/s) - ‘wikipedia.csv’ saved [1003477941/1003477941]



[Row(_c0=0, title='economics', summary='economics () is a social science that studies the production, distribution, and consumption of goods and services.economics focuses on the behaviour and interactions of economic agents and how economies work. microeconomics analyzes what\'s viewed as basic elements in the economy, including individual agents and markets, their interactions, and the outcomes of interactions. individual agents may include, for example, households, firms, buyers, and sellers. macroeconomics analyzes the economy as a system where production, consumption, saving, and investment interact, and factors affecting it: employment of the resources of labour, capital, and land, currency inflation, economic growth, and public policies that have impact on these elements. other broad distinctions within economics include those between positive economics, describing "what is", and normative economics, advocating "what ought to be"; between economic theory and applied economics; b

Mostro che i valori della colonna "categoria" vengano selezionati correttamente

In [3]:
df.select("categoria").distinct().show(50, truncate=False)

+-----------+
|categoria  |
+-----------+
|finance    |
|medicine   |
|research   |
|technology |
|energy     |
|transport  |
|politics   |
|culture    |
|science    |
|humanities |
|economics  |
|trade      |
|sports     |
|pets       |
|engineering|
+-----------+



**Controllo della struttura del Dataset**:

---
Verifico la struttura del dataset e la tipologia di valori per ogni field


In [4]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- documents: string (nullable = true)
 |-- categoria: string (nullable = true)



Rinomino la colonna _c0 in id

In [5]:
# Rinomino la prima colonna _c0 in id
df = df.withColumnRenamed("_c0", "id")

In [6]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- documents: string (nullable = true)
 |-- categoria: string (nullable = true)



##**1.2 Pulizia dei Dati**

**Verifica valori mancanti**:  

---
In tutto ci sono 928 valori nulli in summary e 928 valori nulli in documents, come riportato nell'output sotto codice.

In [7]:
#Controllo Struttura del dataset
for c in df.columns:
  print(f"Numero valori nulli in {c}: {df.filter(col(c).isNull()).count()}")

Numero valori nulli in id: 0
Numero valori nulli in title: 0
Numero valori nulli in summary: 928
Numero valori nulli in documents: 928
Numero valori nulli in categoria: 0


**Verifico i duplicati**: Non ci sono valori duplicati, dato che il numero di righe totali nel df è uguale al numero di righe distinte



In [8]:
#Controllo duplicati
df.count(), df.distinct().count()

(153232, 153232)

**Rimuovo righe senza testo**:  

---
Mantengo solo le righe che hanno un vero testo
(presenza dati sia in cella "`documents`" che in "`summary`")

In [9]:
df = df.filter(col("documents").isNotNull())
df= df.filter(col("summary").isNotNull())

Vado a filtrare le colonne "documents" e "summary" che hanno campo vuoto

In [10]:
df= df.filter(col("documents") != "")
df= df.filter(col("summary") != "")

**Rimuovo categorie vuote**:


---
Elimino tutte le righe con campo categoria vuota. In realtà non ci sarebbe alcun campo con categoria vuota ma lo faccio solo per sicurezza.


In [11]:
df= df.filter(col("categoria").isNotNull())
df= df.filter(col("categoria") != "")

Poichè l'obiettivo del progetto è analizzare e classificare contenuti testuali, rimuoviamo le righe prove di testo.

**Creazione colonna Text**:


---
Creo una colonna `text` che unisce `summary` e `documents`. EQuesto mi permette di avere un unico campo di testo per l’articolo, semplificando le fasi successive d'analisi esplorativa e di addestramento nel modello di classificazione.


In [12]:
from pyspark.sql.functions import concat_ws

In [13]:
df= df.fillna({"summary": ''})
df = df.withColumn("text", concat_ws(" ", col("summary"), col("documents")))

In [ ]:
#Materializzo il DF in modo da evitare che Spark risalga al csv
df=df.cache()
df.count()

facendo prima il `df.cache()` lo conservo in memoria il dataframe per usi successivi (come nell'azione del `df.count()`)

**Controllo Finale**

In [ ]:
print(f"Numero righe rimaste: {df.count()}")
print(f"Numero colonne rimaste: {len(df.columns)}")
df.show(5)

Dopo la pulizia il dataset contiene 152304 righe e 6 colonne. </br>
Ora procedo con l'analisi descrittiva delle categorie.  

#**2. ANALISI DESCRITTIVA (EDA)**

Il primo obiettivo del progetto è condurre un'analisi esplorativa dei dati (EDA) per capire le caratteristiche dei contenuti di Wikipedia suddivisi in diverse categorie tematiche, come ad esempio: - Cultura, Economia, Medicina, Tecnologia, Politica, Scienza, e altre.

L'analisi esplorativa prevede: - Il conteggio degli articoli presenti per ogni categoria. - Il numero medio di parole per articolo. - La lunghezza dell'articolo più lungo e di quello più corto per ciascuna categoria. - La creazione di nuvole di parole rappresentative per ogni categoria, per identificare i termini più frequenti e rilevanti.

## **2.1 Conteggio articoli per categoria**

Per iniziare l'analisi esplorativa verifico come sono distribuiti gli articoli tra le diverse categorie tematiche. <br> Calcolo quindi il numero di articoli presenti in ciascuna categoria.

In [ ]:
# Conteggio articoli per categoria
category_counts = df.groupby('categoria').count().sort(col('count').desc())
category_counts.show()

L’output mostra che le categorie più popolose sono *politics*, *engineering*, *science* e *culture*, tutte con oltre 10.000 articoli.

---
Le categorie leggermente meno rappresentate sono *research* e *finance*, che comunque superano le 9.000 unità.

Per visualizzare meglio la distribuzione, trasformo il DataFrame Spark in Pandas e genero un grafico a barre.

In [ ]:
#Per creare il grafico trasformo Spark to Pandas
import pandas as pd
import matplotlib.pyplot as plt

pdf = category_counts.toPandas()
#Prendo solo le prime 15 categorie

pdf_small = pdf.head(15)
plt.figure(figsize=(10,6))
plt.bar(pdf_small['categoria'], pdf_small['count'])
plt.xticks(rotation=45)
plt.title("Conteggio Articoli per Categoria (Top 15)")
plt.xlabel("Categoria")
plt.ylabel("Numero Articoli")
plt.show()


Il grafico conferma che la categoria politics è la più numerosa, mentre finance è la meno popolosa tra le 15 categorie presenti nel dataset.

##**2.2 Numero medio di parole per articolo**

Per capire quanto sono lunghi in media gli articoli di ciascuna categoria, inizio tokenizzando il testo con `RegexTokenizer`, che suddivide la colonna text in una lista di parole. <br> Successivamente calcolo il numero di parole per ogni articolo tramite `size()` e poi la media per categoria.

**Tokenizzazione Spark**

In [ ]:
from pyspark.ml.feature import RegexTokenizer

tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W")
df_tokens_x = tokenizer.transform(df)

In [ ]:
df_tokens_x.printSchema()

In [ ]:
df_tokens_x.show(5)

**Calcolo numero delle parole**

In [ ]:
# Calcolo numero parole
from pyspark.sql.functions import size
df_tokens_x= df_tokens_x.withColumn("num_words", size(df_tokens_x["tokens"]))

In [ ]:
df_tokens_x.select("categoria", "num_words").show(10)

In [ ]:
df_tokens_x.select("categoria").distinct().show(50, truncate=False)

In [ ]:
df_tokens_x.columns

**Calcolo media per categoria**

A questo punto calcolo la media:

In [ ]:
from pyspark.sql.functions import col
avg_words_per_cat = ( df_tokens_x.groupBy("categoria") .avg("num_words") .orderBy(col("avg(num_words)").desc()) )
avg_words_per_cat.show()

Dai risultati emerge che le categorie science e finance hanno articoli mediamente più lunghi (oltre 2000 parole), mentre categorie come pets e sports presentano testi più brevi.

---
Per visualizzare meglio le differenze tra categorie creo un grafico con le prime 5.


**Grafico**

In [ ]:
pdf_avg= avg_words_per_cat.toPandas()
# Prendiamo solo le prime 5 categorie (formato compatibile)
pdf_avg_small = pdf_avg.head(5)

plt.figure(figsize=(12,6))
bars = plt.bar(pdf_avg_small['categoria'], pdf_avg_small['avg(num_words)'],color='blue', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.title("Numero Medio di Parole per Categoria (Top 5)", fontsize=14)
plt.xlabel("Categoria", fontsize=12)
plt.ylabel("Media parole", fontsize=12)

#Aggiungo valori sopra le barre
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval+5, round(yval, 1), ha='center', va='bottom', fontsize=10)

plt.show()


##**2.3 Articolo più lungo e più corto per categoria**

Per ogni categoria voglio identificare l'articolo più lungo e più corto, utilizzando come metrica il numero di parole calcolato nella sezione precedente. <br> Per prima cosa calcolo la lunghezza massima e minima per categoria:

In [ ]:
from pyspark.sql.functions import min, max
lenght_stats = df_tokens_x.groupby("categoria") \
  .agg(
      max("num_words").alias("max_num_words"),
      min("num_words").alias("min_num_words")
  ) \
  .orderBy("max_num_words", ascending=False)

lenght_stats.show()

Successivamente per ottenere l'articolo effettivo, utilizzo uan finestra (row_number) che mi permette di selezionare un solo articolo per categoria, evitando duplicati nel caso in cui più articoli abbiano la stessa lunghezza.

**Articolo più lungo per categoria**

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

windowSpec = Window.partitionBy("categoria").orderBy(desc("num_words"))

longest_articles = (
    df_tokens_x
    .withColumn("rn", row_number().over(windowSpec))
    .filter(col("rn") == 1)
    .select("categoria", "text", "num_words")
)

longest_articles.show(15, truncate=False)


##**2.4 La creazione di nuvole di parole rappresentative per ogni categoria**

**2.4.1 Frequenza delle parole per categoria**

Per mostrare la frequenza di una parola all'interno di una categoria genero una tabella che mostra le parole più frequenti per ciascuan categoria. <br> Quest'analisi è utile perchè permette di:

*  identificare i termini più rappresentativi per ogni tema
*  Confrontare le categorie tra loro
*  Verificare la corretteza semantica dei testi
*  Supportare la fase di modellazione NLP

---
Prima di calcolare le frequenze rimuovo le stopwords (the, of, and, in, to, a, ecc.), altrimenti dominerebbero completamente la classifica.




**Rimozione delle stopwords**

In [ ]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="tokens_clean"
)

df_clean = remover.transform(df_tokens_x)


**Esplosione dei token e normalizzazione**

In [ ]:
from pyspark.sql.functions import explode, col, lower

df_words = df_clean.select(
    col("categoria"),
    explode(col("tokens_clean")).alias("word")
)

# Rendo tutto minuscolo e rimuovo eventuali stringhe vuote
df_words = df_words.filter(col("word") != "").withColumn("word", lower(col("word")))


**Calcolo le frequenze delle parole per categoria**

Ora calcolo la frequenza di ogni parola all'interno di ciascuna categoria.

In [ ]:
word_freq = (
    df_words.groupBy("categoria", "word")
    .count()
    .orderBy(col("count").desc())
)

word_freq.show(20, truncate=False)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, length, col, desc

windowSpec = Window.partitionBy("categoria").orderBy(col("count").desc())

top_words = (
    word_freq
    .withColumn("rn", row_number().over(windowSpec))
    .filter(col("rn") <= 20)
    .filter(length(col("word")) > 3) #filtro le parole con una lunghezza >3
    .orderBy("categoria", "rn")
)


top_words.show(200, truncate=False)

Stampo le prime 10 parole più frequenti per categoria

In [ ]:
#CATEGORIA = Politics
top_words.filter(col("categoria") == "politics").show(10, truncate=False)

In [ ]:
#CATEGORIA = science
top_words.filter(col("categoria") == "science").show(10, truncate=False)

In [ ]:
#CATEGORIA = culture
top_words.filter(col("categoria") == "culture").show(10, truncate=False)

In [ ]:
#CATEGORIA = economics
top_words.filter(col("categoria") == "economics").show(10, truncate=False)

In [ ]:
#CATEGORIA = pets
top_words.filter(col("categoria") == "pets").show(10, truncate=False)

In [ ]:
#CATEGORIA = engineering
top_words.filter(col("categoria") == "engineering").show(10, truncate=False)

Dalla tabella risultante emergono le parole più significative, una volta tolte le stopwords e le parole non significative per quella categoria sono ad esempio:

*   **science**: research, study, laboratory, biology, university
*   **economics**: workers, economic, policy, party, labour
*   **culture**: art, language, film, dance, festival
*   **pets**: species, fish, breed, freshwater, dogs
*   **politics**: party, election, nation, state, parliament



**2.4.2 WordCloud**

---
La wordcloud permette di visualizzare le parole più frequenti associati alla categoria.


Installo la libreria `Wordcloud`

In [ ]:
from wordcloud import WordCloud

In [ ]:
def generate_wordcloud(text, title):
    wc= WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

Utilizzo il Wordcloud perchè mostra le parole + frequenti per categoria. Le parole più frequenti nella categoria considerata appaiono con dimensioni maggiori. Ad esempio le parole + frequenti in `finance`sono company, man, master.

In [ ]:
for cat in df.select("categoria").distinct().limit(4).toPandas()['categoria']:
  text_cat = " ".join(
      df.filter(col("categoria") == cat) .select("text") .toPandas()['text'] .tolist() )

  wc = WordCloud(width=800, height=400, background_color='white').generate(text_cat)
  plt.figure(figsize=(10,5))
  plt.imshow(wc, interpolation='bilinear')
  plt.axis('off')
  plt.title(f"Wordcloud - {cat}")
  plt.show()

##**2.5 Statistiche Finali**

Di seguito faccio un riepilogo generale tutte le statistiche individuate in questa sezione.

In [ ]:
print("### STATISTICHE FINALI ###\n")

import pyspark.sql.functions as F

# Totale occorrenze di parole (dopo rimozione stopwords)
total_words = word_freq.agg(F.sum("count")).collect()[0][0]

# Numero di parole uniche
unique_words = word_freq.select("word").distinct().count()

print(f"-> Totale occorrenze parole (clean): {total_words}")
print(f"-> Parole uniche (clean): {unique_words}")
print(f"-> Frequenza media per parola: {total_words / unique_words:.2f}")

# Top 10 parole globali
print("\n--- TOP 10 PAROLE GLOBALI (dopo stopwords) ---\n")

global_top = (
    top_words
    .groupBy("word")
    .agg(F.sum("count").alias("total_count"))
    .orderBy(F.desc("total_count"))
    .limit(10)
)

global_top.show(truncate=False)

# Statistiche per categoria
print("\n--- STATISTICHE PER CATEGORIA ---\n")

categoria_stats = (
    word_freq
    .groupBy("categoria")
    .agg(
        F.countDistinct("word").alias("unique_words"),
        F.sum("count").alias("total_words"),
        F.avg("count").alias("avg_frequency")
    )
    .orderBy("categoria")
)

categoria_stats.show(truncate=False)

print("### ANALISI COMPLETATA ###")


**2.6 Riassunto EDA**
---
- Il dataset contiene 152304 articoli distribuiti in 15 categorie.
- Le categorie più popolose sono `politics`, `engineering`, `science`, `culture`.
- Le categorie con articoli mediamente più lunghi sono `science` e `finance`.
- Le categorie con articoli più brevi sono `pets` e `transport`.
- Le analisi sulle parole mostrano che ogni categoria ha un vocabolario caratteristico (es. research per science, market per economics, species per pets).
- Le wordcloud confermano visivamente le differenze semantiche tra categorie.


#**3. SVILUPPO DI UN CLASSIFICATORE AUTOMATICO**
---


Il secondo obiettivo è creare un modello di machine learning capace di classificare automaticamente gli articoli in base alla loro categoria. Il sistema di classificazione verrà addestrato utilizzando dati di testo presenti nelle seguenti colonne del dataset: - Sommario (summary): Introduzione breve dell'articolo. - Testo Completo (documents): Contenuto completo dell'articolo.

##**3.1 Preparazione dei dati**

Obiettivo della seguente sezione:
- utilizzo la colonna `text` come input del modello
- trasformo la colonna `categoria` in una variabile numerica `label`
- divido il dataset in train (80%) e test (20%)

In [ ]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

1. Seleziono le colonne del dataframe utili per Llm.

In [ ]:
#Seleziono le colonne utili al modello "text" & "categoria"
df_ml= df.select(col("text"), col("categoria"))
df_ml.show(5, truncate= False)
print(f"Numero di righe del modello: {df_ml.count()}")
print(f"Numero di testi unici: {df_ml.select("text").distinct().count()}")

2. Codifico i valori della colonna categoria in una nuova colonna denominata "label" (cioè trasformo la colonna categoria in numeri)

In [ ]:
label_index= StringIndexer(inputCol="categoria", outputCol="label")


3. Applico l'indexer tramite fit per generare la colonna label

In [ ]:
df_ml_index= label_index.fit(df_ml).transform(df_ml)
df_ml_index.select("text","categoria","label").show(5, truncate= False)

4. Suddivido il dataset tramite train-split in train e test (80% train, 20% test)

In [ ]:
train_df, test_df= df_ml_index.randomSplit([0.8,0.2], seed=42)
print(f"Righe train_df: {train_df.count()}")
print(f"Righe test_df: {test_df.count()}")

##**3.2 Creazione della Pipeline NLP + Classificatore**

In questa sezione preparo la pipeline di trasformazione del testo e definisco il modello di classificazione.
L'obiettivo è:
- trasformare il testo in numeri vettoriali
- rimuovere le stopwords
- applicare IDF (TF-IDF weighting)
- addestrare il classificatore (Logistic Regression)

In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

1. Tokenizzo il testo (RegexTokenizer)

In [ ]:
tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W", toLowercase=True)
#patter= "\\W" -> separa i caratteri non alfanumerici

2. Rimuovo delle stopwords (StopWordsRemover)

In [ ]:
remover= StopWordsRemover(inputCol="tokens", outputCol="tokens_clean")

3. Trasformo le parole in vettori numerici (CountVectorizer)

In [ ]:
vectorizer= CountVectorizer(inputCol="tokens_clean", outputCol="rawFeatures", vocabSize=5000)

4. **IDF**: Calcolo i pesi TF-IDF

In [ ]:
idf= IDF(inputCol="rawFeatures", outputCol="features")

5. **Regressione Logistica**: creo un classificatore con una Regressione Logistica Multiclasse

In [ ]:
lr= LogisticRegression(featuresCol="features", labelCol="label", maxIter= 50)

6. Pipeline completa

In [ ]:
pipeline= Pipeline(stages=[tokenizer, remover, vectorizer, idf, lr])

In [ ]:
pipeline

La pipeline mi permette di eseguire tutte le trasformazioni in modo automatico in sequenza.

##**3.3 Addestramento del modello**

Dopo l'addestramento applico il modello al test-set per ottenere le prime predizioni e verificare che il flusso funzioni correttamente.

In [ ]:
model = pipeline.fit(train_df)

1. Applico il modello al test-set

In [ ]:
test_predictions= model.transform(test_df)

2. Visualizzo alcune predizioni

In [ ]:
test_predictions.select("text","categoria","label","prediction").show(5, truncate= False)

##**3.4 Valutazione del modello**

In questa fase valuto le prestazioni del modello utilizzando il test-set. </br> Le metriche utilizzo sono l'Accuracy, l'F1-score, e la Confusion-Matrix che permettono di capire quanto il modello riesce a distinguere corettamente le categorie.

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",metricName="accuracy")

In [ ]:
evaluator.evaluate(test_predictions)

Ora calcolo l’accuracy per verificare quanto il modello predice correttamente le categorie.

1. Accuracy

In [ ]:
accuracy = evaluator.setMetricName("accuracy").evaluate(test_predictions)
print(f"Accuracy: {accuracy:.4f}")

2. F1-score

In [ ]:
f1 = evaluator.setMetricName("f1").evaluate(test_predictions)
print(f"F1-score: {f1:.4f}")

L'accuracy e l'F1-score mostrano che il modello ha una buona capacità di classificazione (93%). La Confusion Matrix permette di vedere quali categorie vengono riconosciute meglio e quali vengono confuse.

3. Confusion Matrix

In [ ]:
confusion_df = (
    test_predictions .groupBy("categoria", "prediction")
    .count()
    .orderBy("categoria", "prediction") )
confusion_df.show(20, truncate=False)

##**3.5 Predizione su nuovi articoli**

Infine utilizzo il modello appena validato per classificare in automatico un nuovo articolo di Wikipedia. </br>
Quindi dovrò prendere un nuovo articolo, applicare il modello (applicando in auto tokenizzazione, rimozione stopwords, CountVectorizer, TD-IDF e classificazione) e generare la categora predetta

In [ ]:
nuovo_articolo = """ The Saxe–Goldstein hypothesis is a prediction in archaeology about the relationship between a society's funerary practices and its social organization. It predicts a correlation between two phenomena: the use of specific areas to dispose of the dead, and the legitimation of control over restricted resources through claims of descent from dead ancestors. """
# Creo un DataFrame Spark con il nuovo testo
df_nuovo = spark.createDataFrame([(nuovo_articolo, )], ["text"])

In [ ]:
# Applico il modello per ottenere la categoria predetta
predizione = model.transform(df_nuovo)
# Mostro la categoria predetta
predizione.select("text", "prediction").show(truncate=False)

Infine mostro la categoria a livello testuale per mostrare l'effettiva categoria alla quale l'articolo appartiene.

Per ottenere la categoria testuale corrispondente alla predizione numerica, recupero la lista delle etichette originali dal modello di StringIndexer.

In [ ]:
indexer_model = label_index.fit(df_ml)
labels = indexer_model.labels

In [ ]:
#Estraggo la predizione numerica
pred_num= predizione.select("prediction").collect()[0][0]
#Estraggo la categoria corrispondente
print("Categoria predetta: ", labels[int(pred_num)])

Quindi l'articolo riportato appartiene alla categoria *humanities*.

##**3.6 Riassunto Sviluppo di un Classificatore Auto**

---
In questo modulo ho costruito un modello in grado di classificare in automatico gli articoli di Wikipedia in base al contenuto.
Dopo una prima preparazione dei dati, ho generato una pipeline che trasforma il testo (tokenizzazione, rimozione delle stopwords, TF-IDF) e addestra uan Regressione Logistica multiclasse. </br>
Per valutare il modello ho utilizzato tre metriche: accuratezza, F1-score, Confusion Matrix, che mi ha fornito una precisione del modello del 93%.</br>
infine ho testato il modello su un articolo preso da Wikipedia verificando a quale categoria appartiene.


#**4. IDENTIFICAZIONE DI NUOVI INSIGHTS**

---
L'analisi consentirà anche di ottenere preziosi insights sui contenuti di Wikipedia, come la densità di articoli per categoria o le tendenze linguistiche associate a determinati argomenti. Queste informazioni possono aiutare Wikimedia a migliorare l'organizzazione delle pagine e a ottimizzare i propri sforzi editoriali.


##**4.1 Distribuzione degli articoli per categoria**

Per prima cosa verifico quante pagine appartengono a ciascuna categoria

In [ ]:
df.groupBy("categoria").count().orderBy("count", ascending=False).show(truncate=False)

**INSIGHTS**: La distribuzione degli articoli è uniforme: quasi tutte le categorie si trovano sui 10.000 articoli. L'unica eccezione è *politics* che risulta leggermente sopra-rappresentata mentre finance e research hanno valori leggermente inferiori ma vicini alla media.

##**4.2 Lunghezza media degli articoli per categoria**

Calcolo la lunghezza media del testo per categoria per capire quali argomenti presentano descrizioni più estese.

In [ ]:
from pyspark.sql.functions import length, avg

df_len= df.withColumn("text_lenght",length("text"))
df_len.groupBy("categoria").agg(avg("text_lenght").alias("lunghezza_media")).orderBy("lunghezza_media", ascending=False).show(truncate=False)

**INSIGHTS**: La lunghezza degli articoli varia in modo significatico tra le vaire categorie. Le aree *science* e *finance* risultano le più estese , con testi ch superano in media le 12.000 parole. Al contrario categorie come *pets*, *sports*, *energy* hanno testi più brevi e sintetici. Questo è dovuto alla presenza in queste categorie dalla natura degli argomenti con descrizioni meno tecniche o meno approfondite. </br>
In generale le categorie più tecniche tendono ad avere articoli più lunghi mentre quelle pratiche hanno articoli più brevi.

##**4.3 Parole più frequenti per categoria**

Analizzo le parole più frequenti per categoria per individuare eventuali pattern linguistici.

In [ ]:
top_n= 10
for categoria in word_freq.select("categoria").distinct().rdd.flatMap(lambda x: x).collect():
  print(f"\n--- Top {top_n} parole per la categoria: {categoria} ---")

  top_words_cat = (
      word_freq
      .filter(word_freq.categoria == categoria)
      .orderBy("count", ascending=False) .limit(top_n) )

  top_words_cat.select("word", "count").show(truncate=False)

**INSIGHTS** : Ogni categoria mostra un vocabolrio molto distinto. Le aree tecniche (science, energy, engigneering) utilizzano termini specialistici, mentre culture e humanities hanno un linguaggio + narrativo. Le categorie economiche e politiche presentano invece un lessico istituzionale e sociale. </br>
Queste differenze confermano che gli argomenti trattati influenzano fortemente lo stile linguistico degli articoli.

##**4.4 Vocabolario per Categoria**

Calcolo quante parole diverse compaiono in ciascuan categoria

In [ ]:
from pyspark.sql.functions import countDistinct
df_words.groupBy("categoria").agg(countDistinct("word").alias("parole_distinte")).orderBy("parole_distinte", ascending=False).show(truncate= False)

**INSIGHTS** : Le categorie con il vocabolario più ampio sono medicine, humanities e research, che superano le 140k parole. Segno di testi ricchi e molto vari. Al contrario politics è la cateogria con meno parole diverse, mentre sports, pets ed energy mostrano una varietà linguistica limitata.</br>
In generale le aree più tecniche e accademiche tendono avere un linguaggio più ampio mentre quelle più tematiche e descrittive risultano più ripetitive.

##**4.6 Considerazioni Finali**

---
Guardando gli articoli per categoria, il dataset risulta bilanciato: quasi tutte le aree hanno circa 10.000 articoli, quindi non ci sono squilibri perticolari. </br>
La lunghezza dei testi invece cambia parecchio. Science, finance e politics hanno articoli molto più lunghi della media, mentre categorie come pets, sports ed energy sono più brevi e dirette. </br>
Le parole più frequenti confermano che ogni categoria a una sua terminologia: termini scientifici nelle aree scientifiche, linguaggio istituzionale in politics, parole legate a software o sistemi in tecnology e un tono più narrativo in culture e humanities. </br>
Anche il vocaolario unico varia molto. le categorie più accademiche (medicine, humanities, research) usano un numero molto alto di diverse parole, a differena di politics o categorie più tematiche.  </br>
In sintesi, pur essendo bilanciate al numero di articoli, le categorie differiscono molto per stile, lunghezza e varietà del linguaggio. Queste differenze aiutano a capire come sono strutturato i contenuti e dove ci potrebbero essere margini di miglioramento.


#**CONCLUSIONI**

---

Il progetto offre a Wikimedia un potente strumento di analisi dati e classificazione automatica per migliorare la gestione dei propri contenuti. Attraverso l'utilizzo di tecniche avanzate di data science e machine learning, Wikimedia sarà in grado di ottimizzare la propria infrastruttura informativa e offrire un servizio di qualità superiore agli utenti di tutto il mondo.